# 第5回　連続確率分布：正規分布の数理
## ―― なぜ世界は正規分布だらけなのか。そして「すべてが正規」ではない

統計学Ⅱ　2026後期　／　北星学園大学　／　小野原 彩香

---

### このノートの使い方

統計学Ⅰでは「正規分布は便利な **仮定** だ」と学んだ。Ⅱでは一歩進んで、**なぜ正規分布がこれほど現れるのか** を数理で解く。答えはまた **独立** にある。▶ を上から押そう。

In [ ]:
# 準備：ライブラリと、霊長類376種のデータ（§4で使う）。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats

def _build_from_source():
    """公開データ（PanTHERIA）から、この授業で使う形に組み立て直す。"""
    # 原典（PanTHERIA）から組み立て直す。まずリポジトリ同梱の複製、だめなら発行元から。
    # 発行元は User-Agent を見て弾くことがあるため、明示して取得する。
    import io, urllib.request
    SRCS = [
        "https://raw.githubusercontent.com/aonoa68/toukei-2/main/docs/data/PanTHERIA_1-0_WR05_Aug2008.txt.gz",
        "https://esapubs.org/archive/ecol/E090/184/PanTHERIA_1-0_WR05_Aug2008.txt",
    ]
    fam = {"Cercopithecidae":"オナガザル科","Cebidae":"オマキザル科","Pitheciidae":"サキ科",
           "Atelidae":"クモザル科","Cheirogaleidae":"コビトキツネザル科","Lemuridae":"キツネザル科",
           "Galagidae":"ガラゴ科","Hylobatidae":"テナガザル科","Indriidae":"インドリ科",
           "Lorisidae":"ロリス科","Lepilemuridae":"イタチキツネザル科","Aotidae":"ヨザル科",
           "Hominidae":"ヒト科","Tarsiidae":"メガネザル科","Daubentoniidae":"アイアイ科"}
    cols = {"MSW05_Binomial":"学名","MSW05_Genus":"属","5-1_AdultBodyMass_g":"体重g",
            "13-1_AdultHeadBodyLen_mm":"頭胴長mm","5-3_NeonateBodyMass_g":"新生児体重g",
            "10-2_SocialGrpSize":"集団サイズ","9-1_GestationLen_d":"妊娠期間日",
            "25-1_WeaningAge_d":"離乳日齢","3-1_AgeatFirstBirth_d":"初産日齢",
            "14-1_InterbirthInterval_d":"出産間隔日","15-1_LitterSize":"一腹産子数",
            "17-1_MaxLongevity_m":"最長寿命月","22-1_HomeRange_km2":"行動圏km2",
            "21-1_PopulationDensity_n/km2":"個体群密度","26-1_GR_Area_km2":"分布域km2",
            "6-2_TrophicLevel":"栄養段階","12-1_HabitatBreadth":"生息環境幅",
            "28-2_Temp_Mean_01degC":"平均気温01","28-1_Precip_Mean_mm":"月降水量mm"}
    src = None
    for _url in SRCS:
        try:
            _req = urllib.request.Request(_url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(_req, timeout=60) as _r:
                _raw = _r.read()
            _comp = "gzip" if _url.endswith(".gz") else None
            src = pd.read_csv(io.BytesIO(_raw), sep="\t", compression=_comp)
            break
        except Exception:
            continue
    if src is None:
        raise RuntimeError("原典データを取得できませんでした")
    p = src[src["MSW05_Order"] == "Primates"]
    out = p[list(cols)].rename(columns=cols)
    out.insert(1, "科", p["MSW05_Family"].map(fam))
    t = out.pop("平均気温01")
    out["平均気温C"] = np.where(t == -999, -999, (t / 10).round(1))
    return out.replace(-999, np.nan).sort_values("学名").reset_index(drop=True)

try:
    df = pd.read_csv("https://aonoa68.github.io/toukei-2/data/primates.csv")
except Exception:
    df = _build_from_source()

print("種数:", len(df), " 科数:", df["科"].nunique())
df.head()

---
## 1. 正規分布の形 ―― μ と σ

正規分布 $N(\mu,\sigma^2)$ の確率密度関数は、この式で決まる。

$$f(x)=\frac{1}{\sqrt{2\pi}\,\sigma}\exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

難しく見えるが、形を支配するのは2つだけ ―― **中心 $\mu$（平均）** と **広がり $\sigma$（標準偏差）**。
$\mu$ は山の位置、$\sigma$ は山の幅を決める。実際に動かしてみよう。

In [ ]:
xs = np.linspace(-10, 15, 500)
plt.figure(figsize=(7, 4))
for μ, σ, c in [(0, 1, "#3949ab"), (0, 2, "#4dabb6"), (4, 1, "#e8503a")]:
    plt.plot(xs, stats.norm(μ, σ).pdf(xs), color=c, label=f"μ={μ}, σ={σ}")
plt.xlabel("x")
plt.ylabel("確率密度 f(x)")
plt.title("正規分布：μ が山の位置、σ が山の幅を決める")
plt.legend()
plt.show()

---
## 2. 連続分布では「面積」が確率

離散分布（第4回）では「ちょうど $k$ 回」の確率を棒の高さで表した。
連続分布では、**1点の確率は 0**。意味があるのは「ある範囲に入る確率」＝ **密度曲線の下の面積** だ。

$$P(a \le X \le b) = \int_a^b f(x)\,dx \quad(\text{曲線の下の面積})$$

正規分布で有名なのが **68–95–99.7 則**：平均から $\pm1\sigma$ に約68%、$\pm2\sigma$ に約95%、$\pm3\sigma$ に約99.7% が入る。面積で確かめよう。

In [ ]:
分布 = stats.norm(0, 1)
for k in [1, 2, 3]:
    面積 = 分布.cdf(k) - 分布.cdf(-k)
    print(f"平均 ±{k}σ に入る確率 = {面積:.4f}（約 {面積:.1%}）")

xs = np.linspace(-4, 4, 500)
plt.figure(figsize=(7, 4))
plt.plot(xs, 分布.pdf(xs), color="#333")
for k, c in [(1, "#3949ab"), (2, "#4dabb6"), (3, "#e8d24d")]:
    mask = (xs >= -k) & (xs <= k)
    plt.fill_between(xs[mask], 分布.pdf(xs[mask]), alpha=0.25, color=c)
plt.title("68–95–99.7 則：±1σ=68%, ±2σ=95%, ±3σ=99.7%")
plt.xlabel("標準化した値 z")
plt.ylabel("確率密度")
plt.show()

**標準化**：どんな正規分布も $z=\dfrac{x-\mu}{\sigma}$ で「平均0・標準偏差1」の標準正規分布に直せる。

偏差値はこれの応用で、$\text{偏差値}=50+10z$。テストの点を、平均・ばらつきの違うテスト間で比べられるようにする道具だ。

---
## 3. なぜ正規分布が現れるのか ―― 独立な要因の「和」

身長、測定誤差、テストの点……世の中には正規分布に近いものが多い。なぜか。

答え：**多数の小さな独立な要因が足し合わさると、その合計は正規分布に近づく**から。身長なら何百もの遺伝子と環境要因の足し算、というように。

実験してみよう。$[0,1]$ の一様乱数（まったく正規ではない、真っ平らな分布）を、**何個も独立に足す** と何が起きるか。

In [ ]:
rng = np.random.default_rng(5)
試行 = 100_000
足す個数リスト = [1, 2, 5, 30]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, k in zip(axes, 足す個数リスト):
    合計 = rng.random((試行, k)).sum(axis=1)
    ax.hist(合計, bins=50, density=True, color="#bcd", edgecolor="none")
    # 同じ平均・分散の正規分布を重ねる
    μ, σ = 合計.mean(), 合計.std()
    xs = np.linspace(合計.min(), 合計.max(), 200)
    ax.plot(xs, stats.norm(μ, σ).pdf(xs), color="#e8503a", lw=2)
    ax.set_title(f"一様乱数を {k} 個 足す")
    ax.set_yticks([])
axes[0].set_ylabel("密度")
plt.suptitle("独立な要因を足すほど、合計は正規分布（赤）に近づく")
plt.tight_layout()
plt.show()

1個では真っ平ら、2個で三角、5個でもう釣鐘型、30個では正規分布（赤）にぴたりと重なる。

**これが正規分布が至るところに現れる理由**であり、次回（第6回）の **中心極限定理** の正体でもある。鍵はやはり **「独立に足す」** こと。要因が独立でなければ、この収束は起きない。

---
## 4. だが「すべてが正規」ではない

正規分布は強力だが、**世の中のデータがすべて正規だと思い込むのは誤り**だ。

**生き物の体重**・世帯年収・都市の人口・SNSのフォロワー数・地震の規模などは、**少数の極端に大きな値**を持つ **裾の重い分布（べき分布・対数正規など）** に従う。これらは「独立な要因の**足し算**」ではなく「**掛け算的**・自己強化的」なメカニズムで生まれるため、正規にならない。

> §2〜3で見たとおり、正規分布が現れるのは**独立な効果が足し合わさる**とき。
> 掛け算で効いてくる世界では、足し算の定理は効かない。

シミュレーションではなく、**統計学Ⅰでも使った霊長類376種の実データ**で確かめよう。

In [ ]:
rng = np.random.default_rng(55)
正規データ = rng.normal(500, 100, 100_000)      # 例：測定値のイメージ（独立な効果の足し算）
体重 = df["体重g"].dropna().values               # 実データ：霊長類265種の体重

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(正規データ, bins=60, color="#3949ab", edgecolor="none")
axes[0].axvline(np.mean(正規データ), color="#e8503a", label="平均")
axes[0].axvline(np.median(正規データ), color="k", ls="--", label="中央値")
axes[0].set_title("正規分布（シミュレーション）：平均と中央値がほぼ一致")
axes[0].legend()

axes[1].hist(体重, bins=60, color="#e8503a", edgecolor="none")
axes[1].axvline(np.mean(体重), color="#3949ab", label="平均")
axes[1].axvline(np.median(体重), color="k", ls="--", label="中央値")
axes[1].set_title("霊長類の体重（実データ）：少数の巨大値が平均を吊り上げる")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"正規（シミュ）： 平均 {正規データ.mean():>8,.0f} / 中央値 {np.median(正規データ):>8,.0f}   歪度 {stats.skew(正規データ):+.2f}")
print(f"体重（実データ）： 平均 {体重.mean():>8,.0f} / 中央値 {np.median(体重):>8,.0f}   歪度 {stats.skew(体重):+.2f}")
print()
print(f"最大は {df.loc[df['体重g'].idxmax(), '学名']} の {体重.max():,.0f} g（平均の {体重.max()/体重.mean():.0f} 倍）")

裾の重い分布では、**少数の巨大な値が平均を吊り上げ、平均 ＞ 中央値** になる（第1回でやった「平均年収の罠」そのものだ）。

こういうデータに正規分布を当てはめて「平均±2σ で95%」などと計算すると、**現実離れした結論**になる。**まず分布の形を確かめる**習慣が、統計的誤用を防ぐ第一歩だ。

---
## 今日のまとめ

| ポイント | 中身 |
|---|---|
| 正規分布の形 | 中心 $\mu$ と広がり $\sigma$ だけで決まる |
| 連続分布の確率 | 1点の確率は0。**面積**が確率（68–95–99.7則） |
| 正規が現れる理由 | **多数の独立な要因の和**は正規に近づく（→第6回CLT） |
| 正規でない世界 | 年収・人口などは裾が重い。正規を無条件に仮定しない |

- 正規分布は「独立な和」から生まれる。鍵はまた **独立**。
- すべてが正規ではない。**分布の形を確かめてから**手法を選ぶ。

> **課題（Moodle）**：標準化・確率（面積）の計算（自動採点）＋「正規分布が現れる条件は何か。現れない例を1つ挙げ、なぜ正規にならないか」の記述。詳しくはMoodleの第5回課題を見ること。

> **次回予告**：第6回「標本分布と中心極限定理の再考」。今日見た『独立な和は正規になる』が、標本平均に効く。そして **その魔法には“独立”という隠れた前提がある** ことを暴く ―― 今期の核心へ向かう。